# Plant Disease Classifier: exploration and evaluation

This notebook is the experiment-facing layer. Dataset splitting, transforms, model
architectures, training, image-quality checks, prediction, and artifact creation live
in the reusable `src/` package modules.

The raw PlantVillage files are read in place; this notebook does not copy or modify
the dataset. The test split is reserved for the final evaluation after validation-only
model selection.

In [ ]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
from sklearn.metrics import classification_report, confusion_matrix
from torchvision.utils import make_grid

project_root = Path.cwd().resolve()
if not (project_root / "configs" / "train_config.yaml").is_file():
    project_root = project_root.parent
if not (project_root / "configs" / "train_config.yaml").is_file():
    raise FileNotFoundError("Could not locate configs/train_config.yaml from the notebook")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.data.loaders import build_dataloader, create_plant_village_splits
from src.inference.artifacts import save_model_artifact
from src.inference.predict import DiseasePredictor, predict_tensor
from src.inference.quality import (
    ImageQualityGate,
    denormalize,
    evaluate_with_gate,
    summarise_gate,
)
from src.models.model import TinyVGGClassifier, build_model
from src.models.train import (
    accuracy_fn,
    count_parameters,
    eval_model,
    fit,
    load_config,
    set_seed,
)

config = load_config(project_root / "configs" / "train_config.yaml")
seed = config.get("seed", 42)
set_seed(seed)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

dataset_root = Path(os.environ.get("PLANTVILLAGE_PATH", config["data_root"]))
if not dataset_root.is_absolute():
    dataset_root = project_root / dataset_root

print(f"Project root: {project_root}")
print(f"Dataset: {dataset_root}")
print(f"Device: {device}")

## Load configuration and prepare reproducible data splits

By default, the dataset is read from `data/raw/PlantVillage`. Set
`PLANTVILLAGE_PATH` to use another class-folder directory.

In [ ]:
splits = create_plant_village_splits(
    root=dataset_root,
    image_size=config.get("image_size", 224),
    resize_size=config.get("resize_size", 256),
    train_size=config.get("train_split", 0.70),
    val_size=config.get("val_split", 0.15),
    seed=seed,
)
loader_options = {
    "batch_size": config.get("batch_size", 32),
    "num_workers": config.get("num_workers", 0),
    "seed": seed,
    "pin_memory": device.type == "cuda",
}
train_loader = build_dataloader(splits.train, shuffle=True, **loader_options)
validation_loader = build_dataloader(splits.validation, **loader_options)
test_loader = build_dataloader(splits.test, **loader_options)

class_counts = pd.Series(splits.train.dataset.targets).value_counts().reindex(
    range(len(splits.class_names)), fill_value=0
)
display(pd.DataFrame({"class": splits.class_names, "training_images": class_counts.values}))
print(
    f"Split sizes: train={len(splits.train)}, "
    f"validation={len(splits.validation)}, test={len(splits.test)}"
)

## Inspect the data

Review class balance and a batch of training images before training. The train split
uses augmentation; validation and test use deterministic preprocessing.

In [ ]:
images, labels = next(iter(train_loader))
preview = denormalize(images[: min(20, len(images))])
plt.figure(figsize=(14, 8))
plt.imshow(make_grid(preview, nrow=5).permute(1, 2, 0))
plt.axis("off")
plt.show()
print("Batch shape:", tuple(images.shape))
print("Labels in preview:", [splits.class_names[index] for index in labels[:5].tolist()])

## Train candidates and select on validation data

MobileNetV3-Small is the deployment candidate. The optional TinyVGG baseline is
included for comparison; candidates are selected by validation loss only.

In [ ]:
model = build_model(
    num_classes=len(splits.class_names),
    backbone=config.get("backbone", "mobilenetv3_small_100"),
    pretrained=config.get("pretrained", True),
    freeze_backbone=config.get("freeze_backbone", False),
).to(device)
total_parameters, trainable_parameters = count_parameters(model)
print(f"MobileNet parameters: {total_parameters:,} total, {trainable_parameters:,} trainable")

loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(
    [parameter for parameter in model.parameters() if parameter.requires_grad],
    lr=config.get("lr", 0.001),
)
frozen_module = (
    model.features
    if config.get("freeze_backbone", False) and hasattr(model, "features")
    else None
)
history = fit(
    model=model,
    train_loader=train_loader,
    val_loader=validation_loader,
    loss_fn=loss_fn,
    optimizer=optimizer,
    epochs=config.get("epochs", 10),
    device=device,
    frozen_module=frozen_module,
    patience=config.get("patience"),
)

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot([row["train_loss"] for row in history], label="train")
plt.plot([row["val_loss"] for row in history], label="validation")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.subplot(1, 2, 2)
plt.plot([row["train_acc"] for row in history], label="train")
plt.plot([row["val_acc"] for row in history], label="validation")
plt.xlabel("Epoch")
plt.ylabel("Accuracy (%)")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
candidates = {"MobileNetV3-Small": model}
if config.get("train_baseline", False):
    baseline = TinyVGGClassifier(num_classes=len(splits.class_names)).to(device)
    baseline_optimizer = torch.optim.SGD(baseline.parameters(), lr=0.01)
    fit(
        model=baseline,
        train_loader=train_loader,
        val_loader=validation_loader,
        loss_fn=loss_fn,
        optimizer=baseline_optimizer,
        epochs=config.get("epochs", 10),
        device=device,
        patience=config.get("patience"),
    )
    candidates["TinyVGG"] = baseline

validation_results = {
    name: eval_model(candidate, validation_loader, loss_fn, device)
    for name, candidate in candidates.items()
}
validation_table = pd.DataFrame(
    {
        name: {
            "loss": result["model_loss"],
            "accuracy": result["model_acc"],
            "macro_f1": result["model_macro_f1"],
        }
        for name, result in validation_results.items()
    }
).T
display(validation_table.sort_values("loss"))
selected_name = validation_table["loss"].idxmin()
selected_model = candidates[selected_name]
print(f"Selected on validation loss: {selected_name}")

## Inspect the image-quality gate on validation data

These summaries help assess the configured thresholds without using test results to
tune them.

In [ ]:
quality_gate = ImageQualityGate(**config.get("quality_gate", {}))
validation_gate_results = evaluate_with_gate(
    selected_model, validation_loader, quality_gate, device
)
validation_gate_summary = summarise_gate(
    validation_gate_results, splits.class_names
)
print(f"Rejected: {validation_gate_summary['rejection_rate']:.2f}%")
print("Accuracy on accepted:", validation_gate_summary["accepted_accuracy"])
print("Accuracy on rejected:", validation_gate_summary["rejected_accuracy"])
display(validation_gate_summary["rejection_by_class"].head(10))

## Final test evaluation

Run this section only after candidate selection. The held-out test split is not used
for training, early stopping, or model selection.

In [ ]:
test_results = eval_model(selected_model, test_loader, loss_fn, device)
print(
    f"Test loss: {test_results['model_loss']:.4f} | "
    f"accuracy: {test_results['model_acc']:.2f}% | "
    f"macro-F1: {test_results['model_macro_f1']:.4f}"
)
print(
    classification_report(
        test_results["y_true"],
        test_results["y_pred"],
        labels=range(len(splits.class_names)),
        target_names=splits.class_names,
        digits=3,
        zero_division=0,
    )
)

matrix = confusion_matrix(
    test_results["y_true"],
    test_results["y_pred"],
    labels=range(len(splits.class_names)),
    normalize="true",
)
plt.figure(figsize=(16, 14))
sns.heatmap(
    matrix,
    xticklabels=splits.class_names,
    yticklabels=splits.class_names,
    cmap="Blues",
    cbar=True,
)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Normalized confusion matrix (test split)")
plt.tight_layout()
plt.show()

test_gate_results = evaluate_with_gate(selected_model, test_loader, quality_gate, device)
test_gate_summary = summarise_gate(test_gate_results, splits.class_names)
print(f"Test photo rejection rate: {test_gate_summary['rejection_rate']:.2f}%")

## Inspect predictions and export the deployment artifact

The artifact stores weights, class names, preprocessing settings, quality-gate
thresholds, and final evaluation metrics so inference does not depend on notebook
state.

In [ ]:
for index in range(min(8, len(splits.test))):
    image, label = splits.test[index]
    prediction, confidence = predict_tensor(
        image,
        selected_model,
        splits.class_names,
        device,
        top_k=1,
    )[0]
    plt.figure(figsize=(3, 3))
    plt.imshow(denormalize(image).permute(1, 2, 0))
    plt.title(
        f"True: {splits.class_names[label]}\nPredicted: {prediction} ({confidence:.0%})"
    )
    plt.axis("off")
    plt.show()

artifact_path = project_root / config.get(
    "checkpoint_path", "models/checkpoints/best_model.pt"
)
selected_backbone = (
    "tiny_vgg"
    if selected_name == "TinyVGG"
    else config.get("backbone", "mobilenetv3_small_100")
)
save_model_artifact(
    model=selected_model,
    path=artifact_path,
    class_names=splits.class_names,
    backbone=selected_backbone,
    image_size=config.get("image_size", 224),
    resize_size=config.get("resize_size", 256),
    quality_gate=quality_gate.to_dict(),
    metrics={
        "test_accuracy": test_results["model_acc"],
        "test_macro_f1": test_results["model_macro_f1"],
    },
    seed=seed,
)
print(f"Saved artifact: {artifact_path}")

sample_path, _ = splits.test.dataset.samples[splits.test.indices[0]]
reloaded_predictor = DiseasePredictor(str(artifact_path), device="cpu")
print("CPU reload check:", reloaded_predictor.predict(str(sample_path), top_k=1))